# Tutorial: Estimating Pressure–Temperature Paths from Garnet Compositional Profiles

This notebook illustrates a workflow for inferring metamorphic pressure–temperature (PT) conditions from measured garnet composition traverses, using the _pyMAGEMin_ toolkit. The approach here combines bulk composition setup, pseudosection calculation, element contour plotting, and PT estimation via numerical minimization. While focused on garnet, the workflow can be adapted for other minerals.

---

## 1. Introduction

Compositional zoning in garnet provides a record of the metamorphic PT evolution experienced by a rock. By matching measured compositional profiles to calculated equilibrium phase diagrams, you can estimate PT conditions at different garnet growth stages.

---

## 2. Dependencies
Install and import the required Python modules for data manipulation, visualization, numerical optimization, and MAGEMin integration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from pyMAGEMin.functions.garnet_growth import *
from pyMAGEMin.functions.bulk_rock_functions import *
import pyMAGEMin
import os

import juliacall
from juliacall import Main as jl, convert as jlconvert

In [ ]:
outputPath = './output/MAGEMin_outputs/'

if not os.path.exists(outputPath):
    os.makedirs(outputPath)

---

## 3. Initialize MAGEMin and Define Bulk Rock Composition

Set the thermodynamic database and define the bulk composition of the system. The bulk composition can be customized as appropriate for the sample under consideration.

In [ ]:
# Select database and initialize MAGEMin
db = "mp"
MAGEMin_data = pyMAGEMin.MAGEMin_C.Initialize_MAGEMin(db, dataset=636, verbose=False)

# Define bulk composition (in molar units)
oxidelist = ['H2O', 'SiO2', 'Al2O3', 'CaO', 'MgO', 'FeO', 'MnO', 'K2O', 'Na2O', 'TiO2', 'O']
X = [6.73, 53.99, 15.91, 1.0, 5.47, 9.9, 0.1, 2.61, 2.53, 0.75, 1.18]
sys_in = 'mol'

# Convert to MAGEMin format
X_converted, oxidelist_converted = pyMAGEMin.MAGEMin_C.convertBulk4MAGEMin(
    jlconvert(jl.Vector[jl.Float64], X),
    jlconvert(jl.Vector[jl.String], oxidelist),
    sys_in, db
)

---

## 4. Construct PT Grid

Generate a pressure–temperature grid for evaluating garnet compositions and visualizing isopleths.

In [ ]:
P = np.linspace(1, 10, 27)  # Pressure (kbar)
T = np.linspace(700, 1000, 30)  # Temperature (°C)
Pgrid, Tgrid = np.meshgrid(P, T, indexing='xy')

---

## 5. Calculate Garnet Fraction and Major Elements on the Grid

Evaluate garnet abundance and major-element composition on the PT grid. If a previous calculation exists, load it; otherwise, compute and save it.

In [ ]:
garnet_calculator = pyMAGEMin.MAGEMin_functions.MAGEMinGarnetCalculator()

def generate_PT_gt_grid(P, T, data, X, oxides, sys_in='mol'):
    return garnet_calculator.generate_2D_grid_gt_elements(P, T, data, X, oxides, sys_in, rm_list=None)

output_csv = f"{outputPath}/file_name.csv"



if os.path.exists(output_csv):
    print('file found, loading grid...')
    gt_grid_data = pd.read_csv(output_csv)
else:
    print('file not found, calculating grid...')
    results = generate_PT_gt_grid(Pgrid.flatten(), Tgrid.flatten(), MAGEMin_data, X_converted, oxidelist_converted, sys_in)
    columns = ["gt_mol_frac", "gt_wt_frac", "gt_vol_frac", "Mg", "Mn", "Fe", "Ca"]
    df = pd.DataFrame(np.column_stack([Pgrid.flatten(), Tgrid.flatten(), *results]), columns=["P", "T"] + columns)
    df.to_csv(output_csv, index=False)
    gt_grid_data = df

---

## 6. Plotting Elemental Isopleths on the PT Grid

Overlay compositional contours (isopleths) for selected elements onto the PT grid for visual comparison.

In [ ]:
def plot_element_contours(Tgrid, Pgrid, gt_mol_frac, elements, levels, labels, colours, ax, linestyles):
    if ax is None:
        fig, ax = plt.subplots()
    mesh = ax.pcolormesh(Tgrid, Pgrid, gt_mol_frac.reshape(Pgrid.shape), cmap='viridis')
    for i, element in enumerate(elements):
        ax.contour(Tgrid, Pgrid, element.values.reshape(Pgrid.shape), levels=[levels[i]], colors=colours[i], linestyles=linestyles[i])
        ax.text(0.95, 0.2 - i*0.05, f'{labels[i]} = {round(levels[i], 2)}', color=colours[i], transform=ax.transAxes, ha='right', va='bottom', fontsize=10)
    plt.colorbar(mesh, ax=ax, label='Garnet Mole Fraction')
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Pressure (kbar)')
    ax.grid(ls=':', alpha=0.5)
    return ax

---

## 7. Minimization: Estimate PT from Measured Garnet Endmember Fractions

Estimate the PT conditions that best fit measured garnet endmember fractions. The objective function minimizes the normed difference between measured and modeled values using the differential evolution algorithm (Storn & Price, 1997).

In [ ]:
def determine_PT_point(X, oxides, data, sys_in, garnet_data, bounds, x0):
    def objective_fn(xy):
        gt_frac, gt_wt, gt_vol, py, alm, spss, gr, kho, out = garnet_calculator.gt_single_point_calc_endmembers(xy[0], xy[1], data, X, oxides, sys_in, rm_list=None)
        # Endmembers: (Fe, Ca, Mg, Mn)
        result = np.array([alm, gr, py, spss])
        return np.linalg.norm((result - garnet_data) / (garnet_data + 1e-5))
    path = [x0]
    def callback_fn(x, convergence):
        path.append(x.copy())
    result = differential_evolution(objective_fn, bounds, x0=x0, maxiter=20, disp=True, tol=1e-3, callback=callback_fn, polish=True)
    return result, path

> **Note:** Confirm the output order of `gt_single_point_calc_endmembers` matches the measured data columns. Endmember fractions should sum to unity and be in Fe, Ca, Mg, Mn order.

---

## 8. Example: Loop Over Measured Profiles and Estimate PT

For each measured profile, estimate the PT point that best fits the observed endmember values. Adjust file paths as needed.

In [ ]:
# Load measured garnet endmember data (example)
measured_df = pd.read_excel('./Example_file_name.xlsx')
em_data = measured_df[['Xalm*', 'Xgrs*', 'Xprp*', 'Xsps*']].iloc[30:40] / 100  # Fe Ca Mg Mn

In [ ]:
x0 = np.array([6.0, 900.0])  # Initial guess (P in kbar, T in °C)
bounds = [(P.min(), P.max()), (T.min(), T.max())]

In [ ]:
elements = [gt_grid_data['Fe'], gt_grid_data['Ca'], gt_grid_data['Mg'], gt_grid_data['Mn']]
labels = ['Fe', 'Ca', 'Mg', 'Mn']
colours = ['r', 'y', 'g', 'b']
linestyles = ['-','--', '-.', ':']

In [ ]:
pt_results = []
path = []
for i, row in em_data.iterrows():
    result, path_res = determine_PT_point(X_converted, oxidelist_converted, MAGEMin_data, sys_in, row.values, bounds, x0)
    pt_results.append(result.x)
    path.append(path_res)
    
    fig, ax = plt.subplots()
    plot_element_contours(Tgrid, Pgrid, gt_grid_data['gt_mol_frac'].values, elements, row.values, labels, colours, ax, linestyles)
    ax.plot(np.array(path_res)[:,1], np.array(path_res)[:,0], zorder=3, c='k')
    ax.scatter(result.x[1], result.x[0], c='r', marker='x', zorder=2)
    # plt.savefig(f"{outputPath}/PT_path_{i}.png", dpi=300)
    plt.show()

---

## 9. Visualize the Estimated PT Path

Plot the points inferred by the minimization procedure, tracing the PT path experienced during garnet growth.

In [ ]:
P_result, T_result = np.array(pt_results).T

fig, ax = plt.subplots()
ax.pcolormesh(Tgrid, Pgrid, gt_grid_data['gt_mol_frac'].values.reshape(Pgrid.shape), cmap='viridis')
ax.scatter(T_result, P_result, c='red', marker='x')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Pressure (kbar)')
ax.grid(True)
plt.show()

---

## 10. References
- Storn, R., & Price, K. (1997). Differential evolution – a simple and efficient heuristic for global optimization over continuous spaces. Journal of Global Optimization, 11(4), 341–359.

---

## Notes

- File paths are relative—adjust as needed for your data.
- The script assumes `pyMAGEMin` is installed and compiled with `MAGEMin_C` backend.
- For clarity, error handling and advanced features (e.g., fractionation, rim compositions) are omitted but can be incorporated as needed.